# Healthcare Data Lakehouse Pipeline

This notebook demonstrates a end-to-end Healthcare Data Pipeline following the **Medallion Architecture** (Bronze and Silver layers) using **PySpark** and **Delta Lake** on Databricks[cite: 3].

### Pipeline Overview
1. **Patients Data:** Loaded from the existing Silver layer[cite: 3].
2. **Doctors Data:** Created, ingested into `bronze_doctors`, cleaned, and written to `silver_doctors`[cite: 3].
3. **Admissions Data:** Created, transformed with date formats and metrics (`length_of_stay`), and saved to both `bronze_admissions` and `silver_admissions`[cite: 3].
4. **Gold/Unified View:** Joins Patients, Doctors, and Admissions to create a unified view of healthcare data[cite: 3].

## Step 1: Load Existing Silver Patients Data
We start by reading the pre-existing patient records from the Silver layer Delta table `workspace.healthcare.silver_patients` and displaying the results[cite: 3].

In [0]:
patients = spark.table("workspace.healthcare.silver_patients")
display(patients)

## Step 2: Inspect Patient Data Schema
We check the schema of the `patients` DataFrame to verify column data types such as `patient_id`, `age`, and `registration_date`[cite: 3].

In [0]:
patients.printSchema()

## Step 3: Import PySpark Row Object
Importing `Row` from `pyspark.sql` allows us to construct mock data records programmatically[cite: 3].

In [0]:
from pyspark.sql import Row

## Step 4: Define Doctors Raw Dataset
We construct a list of `Row` objects containing doctor information including ID, name, department, and specialization[cite: 3].

In [0]:
doctors_data = [
    Row(
        doctor_id="D001",
        doctor_name="Dr. Rajesh Kumar",
        department="Cardiology",
        specialization="Cardiologist"
    ),
    Row(
        doctor_id="D002",
        doctor_name="Dr. Priya Sharma",
        department="Neurology",
        specialization="Neurologist"
    ),
    Row(
        doctor_id="D003",
        doctor_name="Dr. Amit Verma",
        department="Orthopedics",
        specialization="Orthopedic Surgeon"
    ),
    Row(
        doctor_id="D004",
        doctor_name="Dr. Sneha Reddy",
        department="Pediatrics",
        specialization="Pediatrician"
    ),
    Row(
        doctor_id="D005",
        doctor_name="Dr. Arjun Rao",
        department="General Medicine",
        specialization="General Physician"
    )
]

## Step 5: Convert Doctors List to DataFrame
We convert the raw list of rows into a PySpark DataFrame using `spark.createDataFrame()` and inspect the data[cite: 3].

In [0]:
doctors_df = spark.createDataFrame(
    doctors_data
)

display(doctors_df)

## Step 6: Ingest Doctors Raw Data into Bronze Layer
We persist the raw doctor data directly into the Delta Lake Bronze table `workspace.healthcare.bronze_doctors` using `overwrite` mode[cite: 3].

In [0]:
doctors_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.healthcare.bronze_doctors"
    )

## Step 7: Verify Bronze Doctors Table
We query the stored `bronze_doctors` Delta table to ensure that ingestion was successful[cite: 3].

In [0]:
display(
    spark.table(
        "workspace.healthcare.bronze_doctors"
    )
)

## Step 8: Bronze to Silver Transformation Flow
```
Bronze Doctors
      ↓
Cleaning and Transformation
      ↓
Silver Doctors
```
In this phase, we clean string attributes, standardize text formatting, and attach ingestion audit metadata[cite: 3].

## Step 9: Import PySpark Transformation Functions
Importing essential functions for string formatting (`trim`, `initcap`), audit logging (`current_timestamp`), and column referencing (`col`)[cite: 3].

In [0]:
from pyspark.sql.functions import (
    trim,
    initcap,
    current_timestamp,
    col
)

## Step 10: Process Doctors Data for Silver Layer
We apply standardization transformations:
- Trim whitespace and convert `doctor_name` and `department` to title case (`initcap`)[cite: 3].
- Add a `processed_timestamp` column to track data processing time[cite: 3].

In [0]:
doctors_silver = doctors_df \
    .withColumn(
        "doctor_name",
        initcap(
            trim(col("doctor_name"))
        )
    ) \
    .withColumn(
        "department",
        initcap(
            trim(col("department"))
        )
    ) \
    .withColumn(
        "processed_timestamp",
        current_timestamp()
    )

## Step 11: Display Silver Doctors DataFrame
We review the transformed doctors DataFrame to verify string formatting and new metadata columns[cite: 3].

In [0]:
display(doctors_silver)

## Step 12: Save Cleaned Data into Silver Doctors Table
We write the transformed data to the `workspace.healthcare.silver_doctors` Delta table[cite: 3].

In [0]:
doctors_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.healthcare.silver_doctors"
    )

## Step 13: Query and Display Silver Doctors Data
Confirming that the saved Delta table returns clean records directly from storage[cite: 3].

In [0]:
doctors = spark.table(
    "workspace.healthcare.silver_doctors"
)

display(doctors)

## Step 14: Admissions Entity Modeling Pipeline
```
Patient + Doctor + Diagnosis + Admission Information
```
We prepare to construct and ingest hospital admission records linked to our existing patients and doctors[cite: 3].

## Step 15: Retrieve Existing Patient IDs
We extract patient primary keys from the `patients` table to ensure relational integrity with incoming admission records[cite: 3].

In [0]:
patients.select("patient_id").show()

## Step 16: Re-verify Patient Identifiers
Re-confirming `patient_id` values for mapping[cite: 3].

In [0]:
patients.select("patient_id").show()

## Step 17: Define Raw Admissions Data
Creating sample admission entries mapping patient IDs (`P001`-`P005`) to doctor IDs (`D001`-`D005`) alongside admission dates, discharge dates, and diagnoses[cite: 3].

In [0]:
admissions_data = [
    Row(
        admission_id="A001",
        patient_id="P001",
        doctor_id="D001",
        admission_date="2026-01-10",
        discharge_date="2026-01-15",
        diagnosis="Heart Disease"
    ),
    Row(
        admission_id="A002",
        patient_id="P002",
        doctor_id="D002",
        admission_date="2026-01-12",
        discharge_date="2026-01-18",
        diagnosis="Migraine"
    ),
    Row(
        admission_id="A003",
        patient_id="P003",
        doctor_id="D003",
        admission_date="2026-02-05",
        discharge_date="2026-02-10",
        diagnosis="Bone Fracture"
    ),
    Row(
        admission_id="A004",
        patient_id="P004",
        doctor_id="D004",
        admission_date="2026-02-08",
        discharge_date="2026-02-12",
        diagnosis="Viral Fever"
    ),
    Row(
        admission_id="A005",
        patient_id="P005",
        doctor_id="D005",
        admission_date="2026-03-01",
        discharge_date="2026-03-04",
        diagnosis="Diabetes"
    )
]

## Step 18: Convert Admissions Data to DataFrame
Creating `admissions_df` and outputting the dataset to inspect string date fields[cite: 3].

In [0]:
admissions_df = spark.createDataFrame(
    admissions_data
)

display(admissions_df)

## Step 19: Import Date Conversion Functions
Importing `to_date` function to convert string dates to Spark `DateType` objects[cite: 3].

In [0]:
from pyspark.sql.functions import to_date

## Step 20: Cast Date Strings to Proper Date Types
Converting `admission_date` and `discharge_date` columns into PySpark `DateType`[cite: 3].

In [0]:
admissions_clean = admissions_df \
    .withColumn(
        "admission_date",
        to_date(
            col("admission_date")
        )
    ) \
    .withColumn(
        "discharge_date",
        to_date(
            col("discharge_date")
        )
    )

## Step 21: Display Cleaned Admissions DataFrame
We inspect `admissions_clean` to confirm date parsing[cite: 3].

In [0]:
display(admissions_clean)

## Step 22: Business Metric Formula
```
Length of Stay = Discharge Date - Admission Date
```
We calculate patient hospital duration in days[cite: 3].

## Step 23: Import `datediff` Function
Importing `datediff` to calculate day differences between two date columns[cite: 3].

In [0]:
from pyspark.sql.functions import datediff

## Step 24: Derived Attribute: Length of Stay
Creating `length_of_stay` by calculating `datediff(discharge_date, admission_date)`[cite: 3].

In [0]:
admissions_clean = admissions_clean.withColumn(
    "length_of_stay",
    datediff(
        col("discharge_date"),
        col("admission_date")
    )
)

## Step 25: Display Admissions with Length of Stay
Inspecting calculated hospital stay values[cite: 3].

In [0]:
display(admissions_clean)

## Step 26: Add Processing Timestamp
Attaching `processed_timestamp` to mark the Silver layer load time[cite: 3].

In [0]:
admissions_clean = admissions_clean.withColumn(
    "processed_timestamp",
    current_timestamp()
)

## Step 27: Save Raw Admissions to Bronze Layer
Storing unmodified/raw admission records into Delta table `workspace.healthcare.bronze_admissions`[cite: 3].

In [0]:
admissions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.healthcare.bronze_admissions"
    )

## Step 28: Save Processed Admissions to Silver Layer
Writing transformed admission data with `length_of_stay` and timestamps into `workspace.healthcare.silver_admissions`[cite: 3].

In [0]:
admissions_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.healthcare.silver_admissions"
    )

## Step 29: Load Silver Admissions Table
Reading saved Silver admissions back from Delta Lake[cite: 3].

In [0]:
admissions = spark.table(
    "workspace.healthcare.silver_admissions"
)

display(admissions)

## Step 30: Inner Join: Patients and Admissions
We join `patients` and `admissions` on `patient_id` using an `inner` join to align demographic data with clinical admission details[cite: 3].

In [0]:
patient_admissions = patients.join(
    admissions,
    on="patient_id",
    how="inner"
)

display(patient_admissions)

## Step 31: Select Core Patient Admission Attributes
Deduplicating duplicate processing timestamp metadata columns and organizing patient-admission fields[cite: 3].

In [0]:
patient_admissions_selected = patient_admissions.select(
    "patient_id",
    "first_name",
    "last_name",
    "gender",
    "age",
    "city",
    "admission_id",
    "doctor_id",
    "diagnosis",
    "admission_date",
    "discharge_date",
    "length_of_stay"
)

## Step 32: Display Intermediate Patient Admissions Data
Reviewing selected attributes before joining physician details[cite: 3].

In [0]:
display(patient_admissions_selected)

## Step 33: Left Join with Doctors Dataset
Joining the combined `patient_admissions_selected` DataFrame with `doctors` on `doctor_id` using a `left` join to enrich admissions with physician names and departments[cite: 3].

In [0]:
healthcare_joined = patient_admissions_selected.join(
    doctors,
    on="doctor_id",
    how="left"
)

## Step 34: Display Enriched Healthcare Dataset
Viewing the joined output combining patients, admissions, and attending doctor details[cite: 3].

In [0]:
display(healthcare_joined)

## Step 35: Select and Reorder Final Output Columns
We finalize our Gold-layer dataset schema by reordering attributes logically: admission info, patient demographics, attending doctor details, diagnosis, and length of stay[cite: 3].

In [0]:
healthcare_final = healthcare_joined.select(
    "admission_id",
    "patient_id",
    "first_name",
    "last_name",
    "gender",
    "age",
    "city",
    "doctor_id",
    "doctor_name",
    "department",
    "specialization",
    "diagnosis",
    "admission_date",
    "discharge_date",
    "length_of_stay"
)